In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from joblib import load

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay, 
    roc_curve,
    precision_recall_curve,
    auc,
    roc_auc_score, 
    classification_report
)

model = load('../outputs/models/xgboost_tuned_final.pkl')
x_test = pd.read_csv('../data/processed/x_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

THRESHOLD = 0.08

y_scores = model.predict_proba(x_test)[:, 1]
y_preds = (y_scores >= THRESHOLD).astype(int)

print('x_test shape: ', x_test.shape)
print('y_test shape: ', y_test.shape)
print('y_test value counts: \n', y_test.value_counts())
print('Predicted Positives: ', y_preds.sum())

In [ ]:
# Extracting Feature Importances

feature_names = x_test.columns.tolist()
importance_scores = model.feature_importances_

feature_importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_scores
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print(feature_importances_df.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top_n = 15
plot_df = feature_importances_df.head(top_n)

bars = ax.barh(
    plot_df['Feature'][::-1],
    plot_df['Importance'][::-1],
    color='steelblue',
    edgecolor='white'
)

ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title('XGBoost Feature Importances — Top 15 (by Gain)', fontsize=14)
ax.set_xlim(0, plot_df['Importance'].max() * 1.15)

# Add value labels on bars
for bar, val in zip(bars, plot_df['Importance'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_preds)
tn, fp, fn, tp = cm.ravel()

print(f'True Negatives: {tn} - Correctly identified as staying')
print(f'False Positives: {fp} - Incorrectly identified as leaving')
print(f'False Negatives: {fn} - Incorrectly identified as staying')
print(f'True Positives: {tp} - Correctly identifies as leaving')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', linewidths=2, linecolor='white', ax=ax, cbar=False)

business_labels = [
    ['Correctly Identified\nas Staying', 'Flagged as Leaving\n(False Alarms)'],
    ['Missed Leaver\n(Silent Risk)', 'Correctly Flagged\nas Leaving']
]

counts = [
    [tn, fp],
    [fn, tp]
]

# Row-wise percentage of actual classes
row_totals = [tn + fp, fn + tp] # [247, 47]
percentages = [
    [tn / row_totals[0], fp / row_totals[0]],
    [fn / row_totals[1], tp / row_totals[1]]
]

# Font colors — white text on dark cells, dark text on light cells
# Bottom-right (TP) and top-left (TN) are the darkest cells
font_colors = [['grey', 'black'], ['black', 'grey']]

for i in range(2):
    for j in range(2):
        ax.text(
            j + 0.5, i + 0.22,
            business_labels[i][j],
            ha='center', va='center',
            fontsize=9, color=font_colors[i][j],
            fontstyle='italic'
        )
        ax.text(
            j + 0.5, i + 0.52,
            str(counts[i][j]),
            ha='center', va='center',
            fontsize=26, fontweight='bold',
            color=font_colors[i][j]
        )
        ax.text(
            j + 0.5, i + 0.78,
            f'{percentages[i][j]*100:.1f}% of actual',
            ha='center', va='center',
            fontsize=9, color=font_colors[i][j]
        )

ax.set_xlabel('Predicted Label', fontsize=12, labelpad=12)
ax.set_ylabel('Actual Label', fontsize=12, labelpad=12)
ax.set_xticklabels(['Stay (0)', 'Leave (1)'], fontsize=11)
ax.set_yticklabels(['Stay (0)', 'Leave (1)'], fontsize=11, rotation=0)
ax.set_title('Confusion Matrix — XGBoost Tuned v2 @ Threshold 0.08', fontsize=13, pad=15)

plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve and Precision-Recall Curve

fpr, tpr, roc_thresholds = roc_curve(y_test, y_scores)
roc_auc = roc_auc_score(y_test, y_scores)

p, r, pr_thresholds = precision_recall_curve(y_test, y_scores)
pr_auc = auc(r, p)

roc_op_index = np.argmin(np.abs(roc_thresholds - THRESHOLD))
roc_op_fpr = fpr[roc_op_index]
roc_op_tpr = tpr[roc_op_index]

pr_op_index = np.argmin(np.abs(pr_thresholds - THRESHOLD))
pr_op_p = p[pr_op_index]
pr_op_r = r[pr_op_index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ROC Curve ---
ax1 = axes[0]
ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--', label='Random Classifier')
ax1.scatter(roc_op_fpr, roc_op_tpr, color='crimson', s=100, zorder=5, label=f'Operating Point (threshold={THRESHOLD})')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax1.set_title('ROC Curve — XGBoost Tuned v2', fontsize=13)
ax1.legend(loc='lower right', fontsize=10)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.02])

# --- Precision-Recall Curve ---
baseline = y_test.sum() / len(y_test)   # positive class prevalence ~0.16

ax2 = axes[1]
ax2.plot(r, p, color='steelblue', lw=2, label=f'XGBoost (PR-AUC = {pr_auc:.3f})')
ax2.axhline(y=baseline, color='grey', lw=1.5, linestyle='--', label=f'Random Classifier ({baseline:.2f})')
ax2.scatter(pr_op_r, pr_op_p, color='crimson', s=100, zorder=5, label=f'Operating Point (threshold={THRESHOLD})')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve — XGBoost Tuned v2', fontsize=13)
ax2.legend(loc='upper right', fontsize=10)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.02])

plt.suptitle('Model Performance Curves @ Threshold 0.08', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scenario Comparison Table

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

table_data = [
    ['No Model\n(Status Quo)', '0/47', '0%', '0', '247', 'Entirely Reactive'],
    ['Flag Everyone\n(Naive)', '47/47', '100%', '294', '247', 'Impractical'],
    ['XGBoost @ 0.08\n(our Model)', '36/47', '76.6%', '105', '69', 'Actionable-Prioritized List']
]

columns = ['Scenario', 'Leavers\nCaught', 'Recall', 'Total\nFlagged', 'False\nAlarms', 'Verdict']

table = ax.table(
    cellText=table_data,
    colLabels=columns,
    cellLoc='center',
    loc='center'
) 

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.8)

row_colors = ['#f5f5f5', '#f5f5f5', '#e8f4e8']
for i in range(len(columns)):
    table[1, i].set_facecolor(row_colors[0])
    table[2, i].set_facecolor(row_colors[1])
    table[3, i].set_facecolor(row_colors[2])

for i in range(len(columns)):
    table[3, i].set_text_props(fontweight='bold')

ax.set_title('Scenario Comparison — Attrition Prediction Approaches', fontsize=13, pad=20, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/business_impact_table.png', dpi=150, bbox_inches='tight')
plt.show()

    
    

## Business Impact Summary

**What the model does:**
The XGBoost model assigns each employee an attrition risk score. Employees scoring above 0.08 are flagged for proactive HR review.

**Key outcomes on the test set (294 employees):**
- Catches **36 of 47 actual leavers** before they resign (76.6% recall)
- Produces a focused intervention list of **105 employees** (manageable for an HR team vs. reviewing all 294)
- Misses **11 leavers** — unavoidable given the precision-recall tradeoff

**Cost framing (illustrative, assuming $45,000 avg. replacement cost):**
- Potential value if half of flagged true leavers are retained: ~$810,000
- Cost of false alarm interventions: ~$34,500
- Cost of missed leavers: ~$495,000

**Why this threshold?**
In attrition prediction, missing a leaver (False Negative) is significantly more costly than a false alarm (False Positive). Threshold 0.08 was chosen to maximize recall while keeping the intervention list practical.

**What this model cannot do:**
- It cannot guarantee retention — it flags risk, not certainty
- It does not explain *why* an individual is at risk (use feature importance for population-level patterns)
- Performance should be re-evaluated periodically as workforce composition changes